# Notebook 05: Regional Price Divergence Analysis

## Project
NEM Network Constraints & Price Divergence Intelligence System

## Business Question
When do NSW1 and VIC1 prices separate, and can regional price divergence be linked to network constraint activity or interconnector flow behaviour?

## Purpose Of This Notebook
Notebook 05 analyses price divergence between NSW1 and VIC1.

In Notebook 01, we created a clean market context table with price, demand, available generation, and net interchange.

In Notebook 02, we engineered base market features such as price spikes, price change, volatility, and supply-demand gap.

In Notebook 03, we added constraint activity features.

In Notebook 04, we added interconnector flow and congestion-style signal features.

This notebook now compares NSW1 and VIC1 prices directly by creating regional price spread features.


## How This Notebook Links To The Full Project

The central purpose of Project 2 is to explain how network constraints and interconnector flows influence regional price behaviour.

Regional price divergence is one of the clearest market symptoms of network limitations. If NSW1 and VIC1 prices separate materially, it may indicate that electricity cannot flow freely enough between regions to equalise prices.

This notebook supports:

1. Regional price divergence analysis.
2. Network-driven spike classification.
3. Decision intelligence recommendations.
4. Final Power BI dashboard design.

The output from this notebook will be used later to classify whether price spikes were associated with regional separation and congestion-style network conditions.


## Key Market Concepts

### Price spread
Price spread measures the difference between NSW1 and VIC1 RRP at the same dispatch interval.

### Absolute price spread
Absolute price spread measures the size of the regional price difference regardless of direction.

### Divergence flag
A divergence flag identifies intervals where the absolute price spread is materially large.

### Extreme divergence flag
An extreme divergence flag identifies intervals where regional prices are highly separated.

### Analyst interpretation
If NSW1 and VIC1 prices diverge while interconnector or constraint signals are active, this may suggest network-driven price separation.


## Analyst Objective

By the end of this notebook, we want to create NSW1-VIC1 price divergence features:

| Feature | Definition | Formula / Rule | Analyst Meaning |
|---|---|---|---|
| `nsw_rrp` | NSW1 regional reference price. | NSW1 `rrp` | Price signal for NSW1. |
| `vic_rrp` | VIC1 regional reference price. | VIC1 `rrp` | Price signal for VIC1. |
| `price_spread` | NSW1 price minus VIC1 price. | `price_spread = nsw_rrp - vic_rrp` | Positive values mean NSW1 is priced higher than VIC1. Negative values mean VIC1 is priced higher than NSW1. |
| `absolute_price_spread` | Size of the NSW1-VIC1 price difference. | `absolute_price_spread = abs(price_spread)` | Measures the scale of price separation regardless of direction. |
| `divergence_flag` | Material regional price separation. | `absolute_price_spread > 100` | Flags meaningful divergence between NSW1 and VIC1. |
| `extreme_divergence_flag` | Severe regional price separation. | `absolute_price_spread > 300` | Flags intervals where price separation is large enough to require detailed investigation. |


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 120)


In [2]:
PROJECT_ROOT = (
    Path.cwd().parents[0]
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"

MARKET_NETWORK_FILE = OUTPUT_DIR / "04_market_features_with_network.csv"

print("Project root:", PROJECT_ROOT)
print("Input file:", MARKET_NETWORK_FILE)
print("Output directory:", OUTPUT_DIR)


Project root: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence
Input file: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/04_market_features_with_network.csv
Output directory: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs


In [3]:
market_network = pd.read_csv(
    MARKET_NETWORK_FILE,
    parse_dates=["settlementdate"]
)

print("Market network shape:", market_network.shape)

market_network.head()


Market network shape: (16126, 36)


,settlementdate,regionid,rrp,intervention,totaldemand,availablegeneration,netinterchange,intervention_regionsum,supply_demand_gap,price_spike_flag,price_change,rolling_rrp_volatility_1h,date,hour,weekday,is_weekend,active_constraint_count,max_constraint_marginal_value,total_constraint_marginal_value,violation_flag,max_violation_degree,total_absolute_flow,max_absolute_flow,average_absolute_flow,max_absolute_flow_change,high_flow_interconnector_count,high_flow_change_interconnector_count,congestion_signal_count,congestion_signal_flag,vic_nsw_flow,vic_nsw_absolute_flow,vic_nsw_flow_change,vic_nsw_absolute_flow_change,vic_nsw_high_flow_flag,vic_nsw_high_flow_change_flag,vic_nsw_congestion_signal_flag
0,2026-02-01 00:05:00,NSW1,57.05984,0,8186.33,13272.53373,-1054.73,0,5086.20373,False,NaN,NaN,2026-02-01,0,Sunday,True,17,135.3015,-902.14975,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False
1,2026-02-01 00:10:00,NSW1,64.51979,0,8165.29,13246.75420,-924.67,0,5081.46420,False,7.45995,NaN,2026-02-01,0,Sunday,True,16,40.2794,-1006.87256,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False
2,2026-02-01 00:15:00,NSW1,65.08727,0,8202.70,13198.51267,-1006.50,0,4995.81267,False,0.56748,4.479816,2026-02-01,0,Sunday,True,20,9.1600,-1053.75439,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False
3,2026-02-01 00:20:00,NSW1,64.89000,0,8213.41,13169.71135,-1011.73,0,4956.30135,False,-0.19727,3.893369,2026-02-01,0,Sunday,True,20,4.3600,-1061.42631,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False
4,2026-02-01 00:25:00,NSW1,63.47180,0,8055.53,13139.84760,-934.04,0,5084.31760,False,-1.41820,3.381808,2026-02-01,0,Sunday,True,20,4.3600,-1060.65174,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False


## Step 4: Check Available Regions

### What we are doing
We are confirming that NSW1 and VIC1 are both present in the dataset.

### Why it matters
Price divergence requires both regional prices for the same dispatch interval. If one region is missing, the spread calculation will be incomplete.


In [4]:
region_counts = (
    market_network
    .groupby("regionid")
    .agg(
        intervals=("settlementdate", "count"),
        first_interval=("settlementdate", "min"),
        last_interval=("settlementdate", "max"),
        max_rrp=("rrp", "max"),
        average_rrp=("rrp", "mean"),
    )
    .reset_index()
)

region_counts


,regionid,intervals,first_interval,last_interval,max_rrp,average_rrp
0,NSW1,8063,2026-02-01 00:05:00,2026-02-28 23:55:00,20300.00000,83.762187
1,VIC1,8063,2026-02-01 00:05:00,2026-02-28 23:55:00,269.77601,45.055455


## Step 5: Create NSW1 And VIC1 Price Tables

### What we are doing
We are separating NSW1 and VIC1 prices into two small tables with one row per dispatch interval.

### Why it matters
To calculate regional price spread, NSW1 and VIC1 prices must be aligned on the same `settlementdate`.


In [5]:
nsw_price = (
    market_network[market_network["regionid"] == "NSW1"]
    [["settlementdate", "rrp"]]
    .rename(columns={"rrp": "nsw_rrp"})
)

vic_price = (
    market_network[market_network["regionid"] == "VIC1"]
    [["settlementdate", "rrp"]]
    .rename(columns={"rrp": "vic_rrp"})
)

nsw_price.head(), vic_price.head()


(       settlementdate   nsw_rrp
 0 2026-02-01 00:05:00  57.05984
 1 2026-02-01 00:10:00  64.51979
 2 2026-02-01 00:15:00  65.08727
 3 2026-02-01 00:20:00  64.89000
 4 2026-02-01 00:25:00  63.47180,
           settlementdate   vic_rrp
 8063 2026-02-01 00:05:00  47.06047
 8064 2026-02-01 00:10:00  54.34465
 8065 2026-02-01 00:15:00  19.18000
 8066 2026-02-01 00:20:00  19.18000
 8067 2026-02-01 00:25:00  15.18326)

## Step 6: Merge NSW1 And VIC1 Prices

### What we are doing
We are joining NSW1 and VIC1 prices by dispatch interval.

### Why it matters
Price divergence can only be measured when both regions have a price for the same 5-minute interval.


In [6]:
price_divergence = nsw_price.merge(
    vic_price,
    on="settlementdate",
    how="inner"
)

print("Price divergence shape:", price_divergence.shape)

price_divergence.head()


Price divergence shape: (8063, 3)


,settlementdate,nsw_rrp,vic_rrp
0,2026-02-01 00:05:00,57.05984,47.06047
1,2026-02-01 00:10:00,64.51979,54.34465
2,2026-02-01 00:15:00,65.08727,19.18000
3,2026-02-01 00:20:00,64.89000,19.18000
4,2026-02-01 00:25:00,63.47180,15.18326


## Step 7: Create Price Spread Features

### What we are doing
We are calculating the NSW1-VIC1 price spread and the absolute price spread.

### Why it matters
The signed spread tells us which region was priced higher. The absolute spread tells us how large the regional price separation was regardless of direction.

### Formulas
price_spread = nsw_rrp - vic_rrp

absolute_price_spread = abs(price_spread)


In [7]:
price_divergence["price_spread"] = (
    price_divergence["nsw_rrp"] - price_divergence["vic_rrp"]
)

price_divergence["absolute_price_spread"] = (
    price_divergence["price_spread"].abs()
)

price_divergence.head()


,settlementdate,nsw_rrp,vic_rrp,price_spread,absolute_price_spread
0,2026-02-01 00:05:00,57.05984,47.06047,9.99937,9.99937
1,2026-02-01 00:10:00,64.51979,54.34465,10.17514,10.17514
2,2026-02-01 00:15:00,65.08727,19.18000,45.90727,45.90727
3,2026-02-01 00:20:00,64.89000,19.18000,45.71000,45.71000
4,2026-02-01 00:25:00,63.47180,15.18326,48.28854,48.28854


## Step 8: Create Divergence Flags

### What we are doing
We are flagging material and extreme NSW1-VIC1 price divergence intervals.

### Why it matters
These flags identify intervals where regional prices separated enough to require further investigation.

### Rules
- `divergence_flag = True` when absolute price spread is greater than $100/MWh
  
- `extreme_divergence_flag = True` when absolute price spread is greater than $300/MWh


In [8]:
DIVERGENCE_THRESHOLD = 100
EXTREME_DIVERGENCE_THRESHOLD = 300

price_divergence["divergence_flag"] = (
    price_divergence["absolute_price_spread"] > DIVERGENCE_THRESHOLD
)

price_divergence["extreme_divergence_flag"] = (
    price_divergence["absolute_price_spread"] > EXTREME_DIVERGENCE_THRESHOLD
)

price_divergence.head()


,settlementdate,nsw_rrp,vic_rrp,price_spread,absolute_price_spread,divergence_flag,extreme_divergence_flag
0,2026-02-01 00:05:00,57.05984,47.06047,9.99937,9.99937,False,False
1,2026-02-01 00:10:00,64.51979,54.34465,10.17514,10.17514,False,False
2,2026-02-01 00:15:00,65.08727,19.18000,45.90727,45.90727,False,False
3,2026-02-01 00:20:00,64.89000,19.18000,45.71000,45.71000,False,False
4,2026-02-01 00:25:00,63.47180,15.18326,48.28854,48.28854,False,False


## Step 9: Summarise Price Divergence

### What we are doing
We are creating a high-level summary of NSW1-VIC1 price separation.

### Why it matters
This gives a quick view of how often prices diverged and how severe the largest price separation events were.


In [9]:
divergence_summary = pd.DataFrame({
    "intervals": [len(price_divergence)],
    "average_nsw_rrp": [price_divergence["nsw_rrp"].mean()],
    "average_vic_rrp": [price_divergence["vic_rrp"].mean()],
    "average_price_spread": [price_divergence["price_spread"].mean()],
    "average_absolute_price_spread": [price_divergence["absolute_price_spread"].mean()],
    "max_positive_spread_nsw_higher": [price_divergence["price_spread"].max()],
    "max_negative_spread_vic_higher": [price_divergence["price_spread"].min()],
    "divergence_intervals": [price_divergence["divergence_flag"].sum()],
    "extreme_divergence_intervals": [price_divergence["extreme_divergence_flag"].sum()],
})

divergence_summary["divergence_share_pct"] = (
    divergence_summary["divergence_intervals"]
    / divergence_summary["intervals"]
    * 100
)

divergence_summary["extreme_divergence_share_pct"] = (
    divergence_summary["extreme_divergence_intervals"]
    / divergence_summary["intervals"]
    * 100
)

divergence_summary


,intervals,average_nsw_rrp,average_vic_rrp,average_price_spread,average_absolute_price_spread,max_positive_spread_nsw_higher,max_negative_spread_vic_higher,divergence_intervals,extreme_divergence_intervals,divergence_share_pct,extreme_divergence_share_pct
0,8063,83.762187,45.055455,38.706733,44.932888,20308.0,-234.01792,307,60,3.807516,0.74414


## Step 10: Preview Largest Price Divergence Events

### What we are doing
We are listing the dispatch intervals with the largest NSW1-VIC1 absolute price spreads.

### Why it matters
These intervals are the strongest candidates for network-driven price separation and will be reviewed against constraint and interconnector signals.


In [10]:
largest_divergence_events = (
    price_divergence
    .sort_values("absolute_price_spread", ascending=False)
    .head(30)
    .reset_index(drop=True)
)

largest_divergence_events


,settlementdate,nsw_rrp,vic_rrp,price_spread,absolute_price_spread,divergence_flag,extreme_divergence_flag
0,2026-02-05 14:30:00,20300.00000,-8.00000,20308.00000,20308.00000,True,True
1,2026-02-06 14:05:00,19036.53000,13.72000,19022.81000,19022.81000,True,True
2,2026-02-05 17:55:00,9911.42681,56.50000,9854.92681,9854.92681,True,True
3,2026-02-05 18:05:00,9914.02372,119.32582,9794.69790,9794.69790,True,True
4,2026-02-05 16:35:00,9240.83000,8.95000,9231.88000,9231.88000,True,True
5,2026-02-05 14:50:00,9199.30000,-6.76000,9206.06000,9206.06000,True,True
6,2026-02-06 13:35:00,9199.30000,-6.70383,9206.00383,9206.00383,True,True
7,2026-02-07 16:40:00,9199.30000,56.50000,9142.80000,9142.80000,True,True
8,2026-02-05 14:40:00,7380.80815,-8.00000,7388.80815,7388.80815,True,True
9,2026-02-05 18:15:00,1278.88000,100.00000,1178.88000,1178.88000,True,True


## Step 11: Add Direction Of Price Divergence

### What we are doing
We are identifying which region had the higher price during each divergence interval.

### Why it matters
The direction of divergence matters for interpretation. NSW1 higher than VIC1 can indicate different market conditions than VIC1 higher than NSW1.


In [11]:
def classify_divergence_direction(row):
    if row["price_spread"] > 0:
        return "NSW1 higher than VIC1"
    if row["price_spread"] < 0:
        return "VIC1 higher than NSW1"
    return "No price spread"


price_divergence["divergence_direction"] = price_divergence.apply(
    classify_divergence_direction,
    axis=1
)

price_divergence[
    [
        "settlementdate",
        "nsw_rrp",
        "vic_rrp",
        "price_spread",
        "absolute_price_spread",
        "divergence_direction",
    ]
].head()


,settlementdate,nsw_rrp,vic_rrp,price_spread,absolute_price_spread,divergence_direction
0,2026-02-01 00:05:00,57.05984,47.06047,9.99937,9.99937,NSW1 higher than VIC1
1,2026-02-01 00:10:00,64.51979,54.34465,10.17514,10.17514,NSW1 higher than VIC1
2,2026-02-01 00:15:00,65.08727,19.18000,45.90727,45.90727,NSW1 higher than VIC1
3,2026-02-01 00:20:00,64.89000,19.18000,45.71000,45.71000,NSW1 higher than VIC1
4,2026-02-01 00:25:00,63.47180,15.18326,48.28854,48.28854,NSW1 higher than VIC1


## Step 12: Summarise Divergence Direction

### What we are doing
We are summarising how often NSW1 was higher than VIC1 and how often VIC1 was higher than NSW1.

### Why it matters
This helps identify which region was more frequently exposed to higher prices during the study period.


In [12]:
divergence_direction_summary = (
    price_divergence
    .groupby("divergence_direction")
    .agg(
        intervals=("settlementdate", "count"),
        average_price_spread=("price_spread", "mean"),
        average_absolute_price_spread=("absolute_price_spread", "mean"),
        max_absolute_price_spread=("absolute_price_spread", "max"),
        divergence_intervals=("divergence_flag", "sum"),
        extreme_divergence_intervals=("extreme_divergence_flag", "sum"),
    )
    .reset_index()
)

divergence_direction_summary


,divergence_direction,intervals,average_price_spread,average_absolute_price_spread,max_absolute_price_spread,divergence_intervals,extreme_divergence_intervals
0,NSW1 higher than VIC1,7003,48.149812,48.149812,20308.00000,254,60
1,No price spread,11,0.000000,0.000000,0.00000,0,0
2,VIC1 higher than NSW1,1049,-23.928260,23.928260,234.01792,53,0


## Step 13: Join Divergence Features Back To Market Network Dataset

### What we are doing
We are joining NSW1-VIC1 price divergence features back onto the full market-network dataset.

### Why it matters
This gives every NSW1 and VIC1 interval access to the same regional price spread features. Later, spike classification can test whether a high-price interval also occurred during regional price divergence.


In [13]:
divergence_feature_cols = [
    "settlementdate",
    "nsw_rrp",
    "vic_rrp",
    "price_spread",
    "absolute_price_spread",
    "divergence_flag",
    "extreme_divergence_flag",
    "divergence_direction",
]

market_with_divergence = market_network.merge(
    price_divergence[divergence_feature_cols],
    on="settlementdate",
    how="left"
)

print("Market network shape:", market_network.shape)
print("Market with divergence shape:", market_with_divergence.shape)

market_with_divergence.head()


Market network shape: (16126, 36)
Market with divergence shape: (16126, 43)


,settlementdate,regionid,rrp,intervention,totaldemand,availablegeneration,netinterchange,intervention_regionsum,supply_demand_gap,price_spike_flag,price_change,rolling_rrp_volatility_1h,date,hour,weekday,is_weekend,active_constraint_count,max_constraint_marginal_value,total_constraint_marginal_value,violation_flag,max_violation_degree,total_absolute_flow,max_absolute_flow,average_absolute_flow,max_absolute_flow_change,high_flow_interconnector_count,high_flow_change_interconnector_count,congestion_signal_count,congestion_signal_flag,vic_nsw_flow,vic_nsw_absolute_flow,vic_nsw_flow_change,vic_nsw_absolute_flow_change,vic_nsw_high_flow_flag,vic_nsw_high_flow_change_flag,vic_nsw_congestion_signal_flag,nsw_rrp,vic_rrp,price_spread,absolute_price_spread,divergence_flag,extreme_divergence_flag,divergence_direction
0,2026-02-01 00:05:00,NSW1,57.05984,0,8186.33,13272.53373,-1054.73,0,5086.20373,False,NaN,NaN,2026-02-01,0,Sunday,True,17,135.3015,-902.14975,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,57.05984,47.06047,9.99937,9.99937,False,False,NSW1 higher than VIC1
1,2026-02-01 00:10:00,NSW1,64.51979,0,8165.29,13246.75420,-924.67,0,5081.46420,False,7.45995,NaN,2026-02-01,0,Sunday,True,16,40.2794,-1006.87256,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,64.51979,54.34465,10.17514,10.17514,False,False,NSW1 higher than VIC1
2,2026-02-01 00:15:00,NSW1,65.08727,0,8202.70,13198.51267,-1006.50,0,4995.81267,False,0.56748,4.479816,2026-02-01,0,Sunday,True,20,9.1600,-1053.75439,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,65.08727,19.18000,45.90727,45.90727,False,False,NSW1 higher than VIC1
3,2026-02-01 00:20:00,NSW1,64.89000,0,8213.41,13169.71135,-1011.73,0,4956.30135,False,-0.19727,3.893369,2026-02-01,0,Sunday,True,20,4.3600,-1061.42631,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,64.89000,19.18000,45.71000,45.71000,False,False,NSW1 higher than VIC1
4,2026-02-01 00:25:00,NSW1,63.47180,0,8055.53,13139.84760,-934.04,0,5084.31760,False,-1.41820,3.381808,2026-02-01,0,Sunday,True,20,4.3600,-1060.65174,False,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,0.0,0.0,0.0,0.0,False,False,False,63.47180,15.18326,48.28854,48.28854,False,False,NSW1 higher than VIC1


## Step 14: Compare Price Spikes And Divergence

### What we are doing
We are comparing spike and non-spike intervals against NSW1-VIC1 divergence flags.

### Why it matters
This tests whether price spikes occurred during regional price separation. If spike intervals also have high divergence, the spike may be linked to network separation rather than only local demand conditions.


In [14]:
spike_divergence_summary = (
    market_with_divergence
    .groupby(["regionid", "price_spike_flag"])
    .agg(
        intervals=("settlementdate", "count"),
        average_rrp=("rrp", "mean"),
        max_rrp=("rrp", "max"),
        average_price_spread=("price_spread", "mean"),
        average_absolute_price_spread=("absolute_price_spread", "mean"),
        divergence_intervals=("divergence_flag", "sum"),
        extreme_divergence_intervals=("extreme_divergence_flag", "sum"),
        average_active_constraints=("active_constraint_count", "mean"),
        average_congestion_signal_count=("congestion_signal_count", "mean"),
        vic_nsw_congestion_signal_intervals=("vic_nsw_congestion_signal_flag", "sum"),
    )
    .reset_index()
)

spike_divergence_summary


,regionid,price_spike_flag,intervals,average_rrp,max_rrp,average_price_spread,average_absolute_price_spread,divergence_intervals,extreme_divergence_intervals,average_active_constraints,average_congestion_signal_count,vic_nsw_congestion_signal_intervals
0,NSW1,False,8010,68.323133,299.82000,23.193712,29.461064,254,12,19.259051,1.148814,1587
1,NSW1,True,53,2417.098468,20300.00000,2383.221818,2383.221818,53,48,24.415094,1.943396,11
2,VIC1,False,8063,45.055455,269.77601,38.706733,44.932888,307,60,19.292943,1.154037,1598


## Step 15: Create Divergence Event Summary

### What we are doing
We are creating a focused table of intervals where NSW1 and VIC1 prices materially diverged.

### Why it matters
This table helps identify the key price separation events that should be investigated in later spike classification and decision intelligence.


In [15]:
divergence_event_summary = (
    market_with_divergence[
        market_with_divergence["divergence_flag"]
    ]
    [
        [
            "settlementdate",
            "regionid",
            "rrp",
            "nsw_rrp",
            "vic_rrp",
            "price_spread",
            "absolute_price_spread",
            "divergence_direction",
            "extreme_divergence_flag",
            "active_constraint_count",
            "max_constraint_marginal_value",
            "congestion_signal_count",
            "vic_nsw_flow",
            "vic_nsw_absolute_flow",
            "vic_nsw_congestion_signal_flag",
        ]
    ]
    .sort_values("absolute_price_spread", ascending=False)
    .reset_index(drop=True)
)

divergence_event_summary.head(30)


,settlementdate,regionid,rrp,nsw_rrp,vic_rrp,price_spread,absolute_price_spread,divergence_direction,extreme_divergence_flag,active_constraint_count,max_constraint_marginal_value,congestion_signal_count,vic_nsw_flow,vic_nsw_absolute_flow,vic_nsw_congestion_signal_flag
0,2026-02-05 14:30:00,NSW1,20300.00000,20300.00000,-8.00000,20308.00000,20308.00000,NSW1 higher than VIC1,True,24,12.12000,4.0,-1266.00,1266.00,True
1,2026-02-05 14:30:00,VIC1,-8.00000,20300.00000,-8.00000,20308.00000,20308.00000,NSW1 higher than VIC1,True,24,12.12000,4.0,-1266.00,1266.00,True
2,2026-02-06 14:05:00,NSW1,19036.53000,19036.53000,13.72000,19022.81000,19022.81000,NSW1 higher than VIC1,True,23,5.34000,2.0,-20.00,20.00,False
3,2026-02-06 14:05:00,VIC1,13.72000,19036.53000,13.72000,19022.81000,19022.81000,NSW1 higher than VIC1,True,23,5.34000,2.0,-20.00,20.00,False
4,2026-02-05 17:55:00,VIC1,56.50000,9911.42681,56.50000,9854.92681,9854.92681,NSW1 higher than VIC1,True,24,4.96000,2.0,-20.00,20.00,False
5,2026-02-05 17:55:00,NSW1,9911.42681,9911.42681,56.50000,9854.92681,9854.92681,NSW1 higher than VIC1,True,24,4.96000,2.0,-20.00,20.00,False
6,2026-02-05 18:05:00,VIC1,119.32582,9914.02372,119.32582,9794.69790,9794.69790,NSW1 higher than VIC1,True,22,20.23000,2.0,-20.00,20.00,False
7,2026-02-05 18:05:00,NSW1,9914.02372,9914.02372,119.32582,9794.69790,9794.69790,NSW1 higher than VIC1,True,22,20.23000,2.0,-20.00,20.00,False
8,2026-02-05 16:35:00,NSW1,9240.83000,9240.83000,8.95000,9231.88000,9231.88000,NSW1 higher than VIC1,True,29,4.34000,2.0,-20.00,20.00,False
9,2026-02-05 16:35:00,VIC1,8.95000,9240.83000,8.95000,9231.88000,9231.88000,NSW1 higher than VIC1,True,29,4.34000,2.0,-20.00,20.00,False


## Step 16: Create Interval-Level Divergence Event Summary

### What we are doing
We are creating one row per divergence interval instead of one row per region per interval.

### Why it matters
The previous divergence event table includes both NSW1 and VIC1 rows for the same interval. For event review and reporting, it is useful to have a clean interval-level table with one row per dispatch interval.


In [16]:
divergence_interval_summary = (
    price_divergence[price_divergence["divergence_flag"]]
    .merge(
        market_with_divergence[
            [
                "settlementdate",
                "active_constraint_count",
                "max_constraint_marginal_value",
                "total_constraint_marginal_value",
                "violation_flag",
                "max_violation_degree",
                "congestion_signal_count",
                "congestion_signal_flag",
                "vic_nsw_flow",
                "vic_nsw_absolute_flow",
                "vic_nsw_flow_change",
                "vic_nsw_absolute_flow_change",
                "vic_nsw_congestion_signal_flag",
            ]
        ].drop_duplicates(subset=["settlementdate"]),
        on="settlementdate",
        how="left",
    )
    .sort_values("absolute_price_spread", ascending=False)
    .reset_index(drop=True)
)

divergence_interval_summary.head(30)


,settlementdate,nsw_rrp,vic_rrp,price_spread,absolute_price_spread,divergence_flag,extreme_divergence_flag,divergence_direction,active_constraint_count,max_constraint_marginal_value,total_constraint_marginal_value,violation_flag,max_violation_degree,congestion_signal_count,congestion_signal_flag,vic_nsw_flow,vic_nsw_absolute_flow,vic_nsw_flow_change,vic_nsw_absolute_flow_change,vic_nsw_congestion_signal_flag
0,2026-02-05 14:30:00,20300.00000,-8.00000,20308.00000,20308.00000,True,True,NSW1 higher than VIC1,24,12.12000,-100159.89494,False,0,4.0,True,-1266.00,1266.00,232.90,232.90,True
1,2026-02-06 14:05:00,19036.53000,13.72000,19022.81000,19022.81000,True,True,NSW1 higher than VIC1,23,5.34000,-61802.29740,False,0,2.0,True,-20.00,20.00,0.00,0.00,False
2,2026-02-05 17:55:00,9911.42681,56.50000,9854.92681,9854.92681,True,True,NSW1 higher than VIC1,24,4.96000,-22639.48874,False,0,2.0,True,-20.00,20.00,0.00,0.00,False
3,2026-02-05 18:05:00,9914.02372,119.32582,9794.69790,9794.69790,True,True,NSW1 higher than VIC1,22,20.23000,-22761.00366,False,0,2.0,True,-20.00,20.00,0.00,0.00,False
4,2026-02-05 16:35:00,9240.83000,8.95000,9231.88000,9231.88000,True,True,NSW1 higher than VIC1,29,4.34000,-29431.52628,False,0,2.0,True,-20.00,20.00,0.00,0.00,False
5,2026-02-05 14:50:00,9199.30000,-6.76000,9206.06000,9206.06000,True,True,NSW1 higher than VIC1,23,7.45000,-32788.78141,False,0,3.0,True,-505.90,505.90,208.92,208.92,True
6,2026-02-06 13:35:00,9199.30000,-6.70383,9206.00383,9206.00383,True,True,NSW1 higher than VIC1,26,7.99000,-60699.49111,False,0,4.0,True,-966.49,966.49,167.63,167.63,True
7,2026-02-07 16:40:00,9199.30000,56.50000,9142.80000,9142.80000,True,True,NSW1 higher than VIC1,21,10.41000,-20634.31200,False,0,2.0,True,-20.00,20.00,0.00,0.00,False
8,2026-02-05 14:40:00,7380.80815,-8.00000,7388.80815,7388.80815,True,True,NSW1 higher than VIC1,24,9.19000,-22251.99604,False,0,2.0,True,-862.10,862.10,204.41,204.41,True
9,2026-02-05 18:15:00,1278.88000,100.00000,1178.88000,1178.88000,True,True,NSW1 higher than VIC1,21,3.96000,-6279.40831,False,0,2.0,True,-20.00,20.00,0.00,0.00,False


## Step 17: Summarise Network Signals During Divergence

### What we are doing
We are comparing normal intervals, divergence intervals, and extreme divergence intervals.

### Why it matters
This helps test whether price separation was associated with higher constraint activity, higher interconnector congestion-style signal counts, or stronger VIC1-NSW1 flow behaviour.


In [17]:
def divergence_bucket(row):
    if row["extreme_divergence_flag"]:
        return "Extreme divergence"
    if row["divergence_flag"]:
        return "Divergence"
    return "No material divergence"


price_divergence["divergence_bucket"] = price_divergence.apply(
    divergence_bucket,
    axis=1
)

divergence_network_summary = (
    price_divergence[["settlementdate", "divergence_bucket"]]
    .merge(
        market_with_divergence[
            [
                "settlementdate",
                "active_constraint_count",
                "max_constraint_marginal_value",
                "congestion_signal_count",
                "vic_nsw_absolute_flow",
                "vic_nsw_absolute_flow_change",
                "vic_nsw_congestion_signal_flag",
            ]
        ].drop_duplicates(subset=["settlementdate"]),
        on="settlementdate",
        how="left",
    )
    .groupby("divergence_bucket")
    .agg(
        intervals=("settlementdate", "count"),
        average_active_constraints=("active_constraint_count", "mean"),
        max_active_constraints=("active_constraint_count", "max"),
        average_max_constraint_marginal_value=("max_constraint_marginal_value", "mean"),
        average_congestion_signal_count=("congestion_signal_count", "mean"),
        average_vic_nsw_absolute_flow=("vic_nsw_absolute_flow", "mean"),
        average_vic_nsw_absolute_flow_change=("vic_nsw_absolute_flow_change", "mean"),
        vic_nsw_congestion_signal_intervals=("vic_nsw_congestion_signal_flag", "sum"),
    )
    .reset_index()
)

divergence_network_summary


,divergence_bucket,intervals,average_active_constraints,max_active_constraints,average_max_constraint_marginal_value,average_congestion_signal_count,average_vic_nsw_absolute_flow,average_vic_nsw_absolute_flow_change,vic_nsw_congestion_signal_intervals
0,Divergence,247,21.740891,30,61816.454191,2.008097,314.011619,85.288988,45
1,Extreme divergence,60,25.050000,33,12.768473,2.100000,445.656333,133.666833,20
2,No material divergence,7756,19.170449,31,83049.555124,1.119520,520.224447,69.490542,1533


## Step 18: Save Notebook 05 Outputs

### What we are doing
We are saving the price divergence table, event summaries, spike-divergence comparison, and full market dataset with divergence features.

### Why it matters
These outputs become the main input for spike classification and later visualisation. The project now has price, demand, volatility, constraint, interconnector, and divergence features in one dataset.


In [18]:
price_divergence_output = OUTPUT_DIR / "05_regional_price_divergence.csv"
divergence_summary_output = OUTPUT_DIR / "05_divergence_summary.csv"
divergence_direction_summary_output = OUTPUT_DIR / "05_divergence_direction_summary.csv"
divergence_event_summary_output = OUTPUT_DIR / "05_divergence_event_summary.csv"
divergence_interval_summary_output = OUTPUT_DIR / "05_divergence_interval_summary.csv"
spike_divergence_summary_output = OUTPUT_DIR / "05_spike_divergence_summary.csv"
market_with_divergence_output = OUTPUT_DIR / "05_market_features_with_divergence.csv"

price_divergence.to_csv(price_divergence_output, index=False)
divergence_summary.to_csv(divergence_summary_output, index=False)
divergence_direction_summary.to_csv(divergence_direction_summary_output, index=False)
divergence_event_summary.to_csv(divergence_event_summary_output, index=False)
divergence_interval_summary.to_csv(divergence_interval_summary_output, index=False)
spike_divergence_summary.to_csv(spike_divergence_summary_output, index=False)
market_with_divergence.to_csv(market_with_divergence_output, index=False)

print("Saved:", price_divergence_output)
print("Saved:", divergence_summary_output)
print("Saved:", divergence_direction_summary_output)
print("Saved:", divergence_event_summary_output)
print("Saved:", divergence_interval_summary_output)
print("Saved:", spike_divergence_summary_output)
print("Saved:", market_with_divergence_output)


Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/05_regional_price_divergence.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/05_divergence_summary.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/05_divergence_direction_summary.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/05_divergence_event_summary.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/05_divergence_interval_summary.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/05_spike_divergence_summary.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/05_market_features_with_divergence.csv


## Notebook 05 Summary Table

| Step | What We Did | Why It Matters | Output / Feature Created |
|---|---|---|---|
| Step 1-4 | Loaded libraries, project paths, the market-network dataset, and checked available regions. | Confirmed that NSW1 and VIC1 data were available for price comparison. | `market_network`, `region_counts` |
| Step 5 | Created separate NSW1 and VIC1 price tables. | Prepared regional prices for interval-level comparison. | `nsw_price`, `vic_price` |
| Step 6 | Merged NSW1 and VIC1 prices by dispatch interval. | Created one table with both regional prices for each interval. | `price_divergence` |
| Step 7 | Calculated signed and absolute price spread. | Measured both direction and size of NSW1-VIC1 price separation. | `price_spread`, `absolute_price_spread` |
| Step 8 | Created material and extreme divergence flags. | Identified intervals requiring further investigation. | `divergence_flag`, `extreme_divergence_flag` |
| Step 9 | Summarised overall price divergence. | Quantified frequency and severity of NSW1-VIC1 price separation. | `divergence_summary` |
| Step 10 | Previewed largest divergence events. | Highlighted the strongest candidate intervals for network-driven price separation. | `largest_divergence_events` |
| Step 11-12 | Added and summarised divergence direction. | Showed whether NSW1 or VIC1 was more frequently priced higher. | `divergence_direction`, `divergence_direction_summary` |
| Step 13 | Joined divergence features to the market-network dataset. | Added regional spread features to every NSW1 and VIC1 interval. | `market_with_divergence` |
| Step 14 | Compared price spikes and divergence. | Tested whether spike intervals were associated with regional price separation. | `spike_divergence_summary` |
| Step 15-16 | Created region-level and interval-level divergence event summaries. | Produced focused event tables for review, reporting, and later classification. | `divergence_event_summary`, `divergence_interval_summary` |
| Step 17 | Summarised network signals during divergence buckets. | Compared constraint and interconnector signals across normal, divergence, and extreme divergence intervals. | `divergence_network_summary` |
| Step 18 | Saved all Notebook 05 outputs. | Created reusable CSVs for spike classification, visualisation, and Power BI. | `05_*.csv` |
